In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)

seed = 42


sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.10.1
numpy 2.2.4
pandas 2.2.3
sklearn 1.6.1
torch 2.7.0+cpu
cpu


# 加载cifar-10数据集

# 定义resnet模型
模型偏差，层次越深并不代表会表现越好，有可能会出现模型偏差的问题导致表现并不比层次浅的表现好

如果引入了残差，至少保证了每一次更复杂的模型包含了简单的模型，也就是说更复杂的模型至少不会变差

In [2]:
class Residual(nn.Module):
    def __init__(self, inp, oup, use_1x1conv = False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(inp, oup, kernel_size=3, stride=stride, padding=1)
        self.conv2 = nn.Conv2d(oup, oup, kernel_size=3, padding=1)
        if use_1x1conv: # 当输入的层数和输出的层数不同时，用来改输入层的通道的
            self.conv3 = nn.Conv2d(inp, oup, kernel_size=1, stride=stride)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(oup)
        self.bn2 = nn.BatchNorm2d(oup)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3 is not None:
            x = self.conv3(x)
        y += x
        return self.relu(y)

In [3]:
blk = Residual(3, 3)
x = torch.rand(4, 3, 6, 6)
y = blk(x)
print(y.shape)

torch.Size([4, 3, 6, 6])


In [4]:
blk = Residual(3, 6, use_1x1conv = True, stride=2)
blk(x).shape

torch.Size([4, 6, 3, 3])

# 第一个模块的通道数同输入通道数一致。 由于之前已经使用了步幅为2的最大汇聚层，所以无须减小高和宽。 之后的每个模块在第一个残差块里将上一个模块的通道数翻倍，并将高和宽减半。

In [ ]:
def resnet_block(input_channels, num_channels, num_residuals,
                 first_block=False):
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(input_channels, num_channels,
                                use_1x1conv=True, strides=2))
        else:
            blk.append(Residual(num_channels, num_channels))
    return blk

In [5]:
class ResdiualBlock(nn.Module):
    def __init__(self, input_channels, num_channels, num_residuals,
                 first_block=False):
        super().__init__()
        self.model = nn.Sequential()
        for i in range(num_residuals):
            if i == 0 and not first_block:
                self.model.append(Residual(input_channels, num_channels, use_1x1conv=True, stride=2))
            else:
                self.model.append(Residual(num_channels, num_channels))
    
    def forward(self, x):
        return self.model(x)

In [ ]:
class ResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = nn.Sequential(
             nn.Conv2d(
                in_channels=1,
                out_channels=64,
                kernel_size=7,
                stride=2,
            ), 
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            ResdiualBlock(64, 64, 2, first_block=True),
            ResdiualBlock(64, 128, 2),
            ResdiualBlock(128, 256, 2),
            ResdiualBlock(256, 512, 2),
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(512, num_classes),
        )
        self.init_weights()
    
    def init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Conv2d)):
                nn.init.kaiming_uniform_(m.weight) # 何凯明分布
                nn.init.zeros_(m.bias)
        